# Time Series Analysis

**Learning Objectives:**
- Work with datetime in Pandas (`to_datetime()`, `date_range()`)
- Resample time series data with `resample()` (downsampling / upsampling)
- Shift data with `shift()` and `tshift()`
- Compute rolling and expanding window statistics
- Handle time zones, periods, and timedeltas
- Use date offsets

## 1. Setup – Create a DatetimeIndex

Pandas provides rich support for time-based indexing.

In [ ]:
import pandas as pd
import numpy as np

# Generate a range of dates
dates = pd.date_range(start="2025-01-01", periods=30, freq="D")
print("Date range (first 5):", dates[:5])
print("Date range (last 5):", dates[-5:])

## 2. Creating a Time Series DataFrame

We'll simulate stock price data.

In [ ]:
np.random.seed(42)
price = 100.0
prices = []
for _ in range(30):
    price += np.random.randn() * 2
    prices.append(round(price, 2))

stock = pd.DataFrame({
    "Date": dates,
    "Price": prices,
    "Volume": np.random.randint(1000, 10000, 30)
}).set_index("Date")

print("Stock data (first 10 rows):")
print(stock.head(10))

## 3. `to_datetime()` – Parsing Strings to Datetime

Convert string columns to datetime objects.

In [ ]:
# Mixed date formats
raw_dates = pd.Series(["2025-01-15", "Jan 15, 2025", "01/15/2025", "2025-01-15 14:30:00"])
parsed = pd.to_datetime(raw_dates)
print("Parsed dates:")
print(parsed)

## 4. `resample()` – Downsampling (Higher to Lower Frequency)

Change the frequency of the time index, applying an aggregation.

In [ ]:
# Create hourly data
hourly_dates = pd.date_range("2025-01-01", periods=48, freq="h")
hourly = pd.DataFrame({
    "value": np.random.randn(48).cumsum() + 100
}, index=hourly_dates)

# Downsample to daily (mean)
daily = hourly.resample("D").mean()
print("Hourly to Daily (mean):")
print(daily)

In [ ]:
# Resample with multiple aggregations
daily_agg = hourly.resample("D").agg(["mean", "min", "max", "std"])
print(daily_agg)

## 5. `resample()` – Upsampling (Lower to Higher Frequency)

Fill in missing time points. `ffill()` / `bfill()` / `interpolate()` handle gaps.

In [ ]:
# Upsample daily to hourly, forward fill
daily_data = pd.Series([10, 20, 30],
                       index=pd.date_range("2025-01-01", periods=3, freq="D"))
hourly_up = daily_data.resample("h").ffill()
print("Daily to Hourly (ffill, first 10):")
print(hourly_up.head(10))

## 6. `shift()` – Lag / Lead Data

Moves data forward or backward in time.

In [ ]:
stock["Price_Lag1"] = stock["Price"].shift(1)   # previous day
stock["Return"] = stock["Price"].pct_change()   # daily return

print(stock.head(10))

## 7. Rolling Windows – `rolling()`

Apply functions over a sliding window.

In [ ]:
# 7-day rolling mean (moving average)
stock["MA_7"] = stock["Price"].rolling(window=7).mean()
stock["Volatility"] = stock["Price"].rolling(window=7).std()

print(stock[["Price", "MA_7", "Volatility"]].head(10))

In [ ]:
# Rolling with custom function
stock["Range_7"] = stock["Price"].rolling(window=7).apply(lambda x: x.max() - x.min())
print(stock[["Price", "Range_7"]].dropna().head())

## 8. Expanding Windows – `expanding()`

Includes all data from the start up to the current point.

In [ ]:
stock["Expanding_Mean"] = stock["Price"].expanding().mean()
stock["Expanding_Max"] = stock["Price"].expanding().max()

print(stock[["Price", "Expanding_Mean", "Expanding_Max"]].head(10))

## 9. Time Zones and Periods

Pandas supports timezone-aware timestamps and periods.

In [ ]:
# Localize to UTC, then convert to US/Eastern
ts = pd.date_range("2025-01-01", periods=3, freq="h", tz="UTC")
print("UTC:", ts)
print("US/Eastern:", ts.tz_convert("US/Eastern"))

# Period (e.g., monthly)
period = pd.Period("2025-01", freq="M")
print("Period:", period)
print("Next 3 months:", period + pd.offsets.MonthEnd(3))

## 10. Date Offsets

In [ ]:
from pandas.tseries.offsets import BDay, MonthEnd

today = pd.Timestamp("2025-01-15")
print("Today:", today)
print("+5 business days:", today + BDay(5))
print("Month end:", today + MonthEnd(0))

## 11. Practice Problems

In [ ]:
# Problem 1: Compute the 14-day rolling mean and 14-day rolling std
# for the stock Price. Create Bollinger Bands at mean +/- 2*std.
stock["MA_14"] = stock["Price"].rolling(14).mean()
stock["STD_14"] = stock["Price"].rolling(14).std()
stock["Upper_Band"] = stock["MA_14"] + 2 * stock["STD_14"]
stock["Lower_Band"] = stock["MA_14"] - 2 * stock["STD_14"]
print(stock.tail(10))

In [ ]:
# Problem 2: Resample the hourly data to 6-hourly, taking the mean.
# Then compute the cumulative sum.
six_hourly = hourly.resample("6h").mean()
six_hourly["Cumulative"] = six_hourly["value"].cumsum()
print(six_hourly)

In [ ]:
# Problem 3: Create a date_range for all business days in 2025 and
# count how many there are.
biz_days = pd.date_range("2025-01-01", "2025-12-31", freq="B")
print(f"Number of business days in 2025: {len(biz_days)}")

## 12. Summary

| Feature | Purpose |
|---------|---------|
| `to_datetime()` | Parse strings to datetime |
| `date_range()` | Generate date sequences |
| `resample()` | Change frequency (down/up) |
| `shift()` | Lag / lead data |
| `rolling()` | Sliding window statistics |
| `expanding()` | Cumulative statistics |
| `tz_localize/convert` | Timezone handling |
| `offsets` | Business days, month ends, etc. |